In [2]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import os
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score

# Team identifier (replace with your details)
TEAM_NAME = "Vaibhav_13"  # Format: [TeamLeader'sName_HostelNo.]

# Image size
image_size = 224

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Data augmentation and normalization for training
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),  # Random crop and resize
    transforms.RandomHorizontalFlip(),  # Horizontal flip
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),  # Color variation
    transforms.RandomRotation(20),  # Random rotation
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

# Normalization for validation and testing
test_transform = transforms.Compose([
    transforms.Resize(256),  # Resize to 256
    transforms.CenterCrop(image_size),  # Center crop to 224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom Dataset for training and validation
class BirdDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['image_path']
        label = int(self.data.iloc[idx]['label']) - 1  # Shift labels from 1-200 to 0-199
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# Custom Dataset for test images
class TestBirdDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.image_files = [f for f in os.listdir(test_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.test_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_name

# Load and split the dataset
df = pd.read_csv('train_labels.csv')  # Replace with actual path
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)

# Create data loaders
train_dataset = BirdDataset('train_split.csv', transform=train_transform)
val_dataset = BirdDataset('val_split.csv', transform=test_transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)

test_dir = 'data/test'  # Replace with actual test folder path
test_dataset = TestBirdDataset(test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

# Define the CNN architecture
class CustomCNN(nn.Module):
    def __init__(self, num_classes=200):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 224 -> 112
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 112 -> 56
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 56 -> 28
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),  # For 224x224 input
            nn.ReLU(),
            nn.Dropout(0.5),  # Dropout for regularization
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Initialize the model
model = CustomCNN(num_classes=200).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)  # Weight decay for regularization
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)  # Learning rate scheduler

# Training function with metrics
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=5):
    best_bal_acc = 0.0
    patience_counter = 0
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        # Training phase
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        scheduler.step()
        # Validation phase
        model.eval()
        all_labels, all_preds, all_probs = [], [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                probs = torch.softmax(outputs, dim=1)
                _, predicted = torch.max(outputs, 1)
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
        all_labels = np.array(all_labels)
        all_preds = np.array(all_preds)
        all_probs = np.array(all_probs)
        # Compute metrics
        std_acc = (all_preds == all_labels).mean()  # Standard accuracy
        cm = confusion_matrix(all_labels, all_preds)
        per_class_acc = cm.diagonal() / cm.sum(axis=1)
        bal_acc = np.mean(per_class_acc)  # Balanced accuracy
        auc_score = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
        map_score = precision_score(all_labels, all_preds, average='macro', zero_division=0)
        sensitivity = recall_score(all_labels, all_preds, average='macro', zero_division=0)
        specificity = [((cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]) / 
                        (cm.sum() - cm[i, :].sum())) if (cm.sum() - cm[i, :].sum()) > 0 else 0 
                       for i in range(200)]
        avg_specificity = np.mean(specificity)
        # Print results
        print(f"Train Loss: {running_loss / len(train_loader.dataset):.4f} | Std Acc: {std_acc:.4f} | Bal Acc: {bal_acc:.4f}")
        print(f"AUC: {auc_score:.4f} | MAP: {map_score:.4f} | Sensitivity: {sensitivity:.4f} | Specificity: {avg_specificity:.4f}")
        # Early stopping
        if bal_acc > best_bal_acc:
            best_bal_acc = bal_acc
            patience_counter = 0
            torch.save(model.state_dict(), f'best_model_{TEAM_NAME}.pth')
            print("✅ Saved new best model")
        else:
            patience_counter += 1
            print(f"⚠️ Patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("⏹️ Early stopping triggered.")
                break
    return best_bal_acc

# Inference function for test set
def predict_test_images(model, test_loader, device):
    model.eval()
    predictions, image_ids = [], []
    with torch.no_grad():
        for images, img_names in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            predicted = predicted.cpu().numpy() + 1  # Shift back to 1-200
            predictions.extend(predicted)
            image_ids.extend(img_names)
    return pd.DataFrame({'ID': image_ids, 'label': predictions})

# Train the model
best_bal_acc = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=50, patience=5)

# Load best model and generate submission
model.load_state_dict(torch.load(f'best_model_{TEAM_NAME}.pth'))
model.to(device)
submission_df = predict_test_images(model, test_loader, device)
submission_df.to_csv(f'submission_{TEAM_NAME}.csv', index=False)

# Display results
print(f"\nFinal Best Balanced Accuracy: {best_bal_acc:.4f}")
print("Submission preview:")
print(submission_df.head())


Epoch 1/50
Train Loss: 7.5651 | Std Acc: 0.0048 | Bal Acc: 0.0050
AUC: 0.5000 | MAP: 0.0000 | Sensitivity: 0.0050 | Specificity: 0.9950
✅ Saved new best model

Epoch 2/50
Train Loss: 5.2998 | Std Acc: 0.0048 | Bal Acc: 0.0050
AUC: 0.5000 | MAP: 0.0000 | Sensitivity: 0.0050 | Specificity: 0.9950
⚠️ Patience: 1/5

Epoch 3/50
Train Loss: 5.2995 | Std Acc: 0.0048 | Bal Acc: 0.0050
AUC: 0.5000 | MAP: 0.0000 | Sensitivity: 0.0050 | Specificity: 0.9950
⚠️ Patience: 2/5

Epoch 4/50
Train Loss: 5.2993 | Std Acc: 0.0048 | Bal Acc: 0.0050
AUC: 0.5000 | MAP: 0.0000 | Sensitivity: 0.0050 | Specificity: 0.9950
⚠️ Patience: 3/5

Epoch 5/50
Train Loss: 5.2991 | Std Acc: 0.0048 | Bal Acc: 0.0050
AUC: 0.5000 | MAP: 0.0000 | Sensitivity: 0.0050 | Specificity: 0.9950
⚠️ Patience: 4/5

Epoch 6/50
Train Loss: 5.2990 | Std Acc: 0.0048 | Bal Acc: 0.0050
AUC: 0.5000 | MAP: 0.0000 | Sensitivity: 0.0050 | Specificity: 0.9950
⚠️ Patience: 5/5
⏹️ Early stopping triggered.

Final Best Balanced Accuracy: 0.0050
Sub